# 13.5 Terminating a loop: break and continue

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Two other statements work with looping statements to make some scripts easier to
write. The continue statement terminates the current pass through a for or repeat
loop; all further statements in the current pass are skipped, and execution continues with
the test that controls the start of the next pass (if any). The break statement completely
terminates a for or repeat loop, sending control immediately to the statement following
the end of the loop.

As an example of both these commands, [Figure 13-5](./13_5_terminating_a_loop_break_and_continue.ipynb#fig-13-5) exhibits another way of writing
the loop from [Figure 13-4](./13_4_testing_a_condition_the_ifthenelse_statement.ipynb#fig-13-4), so that a `table` entry is made only when there is a change in the
dual value associated with avail[3]. After solving, we test to see if the new dual
value is equal to the previous one:

```ampl
if Time[3].dual = previous_dual then continue;
```

If it is, there is nothing to be done for this value of avail[3], and the continue
statement jumps to the end of the current pass; execution resumes with the next pass,
starting at the beginning of the loop.

After adding an entry to the `table`, we test to see if the dual value has fallen to zero:

```ampl
model steelT.mod;
data steelT.dat;
option solution_precision 10;
option solver_msg 0;
set AVAIL3 default {};
param avail3_obj {AVAIL3};
param avail3_dual {AVAIL3};
let avail[3] := 0;
param previous_dual default Infinity;
repeat {
       let avail[3] := avail[3] + 1;
       solve;
       if Time[3].dual = previous_dual then continue;
              let AVAIL3 := AVAIL3 union {avail[3]};
              let avail3_obj[avail[3]] := Total_Profit;
              let avail3_dual[avail[3]] := Time[3].dual;
              if Time[3].dual = 0 then break;
              let previous_dual := Time[3].dual;
}
display avail3_obj, avail3_dual;
```

**[Figure 13-5](./13_5_terminating_a_loop_break_and_continue.ipynb#fig-13-5):** Using break and continue in a loop (steelT.sa7).

```ampl
if Time[3].dual = 0 then break;
```

If it has, the loop is done and the break statement jumps out; execution passes to the
`display` command that follows the loop in the script. Since the repeat statement in
this example has no while or until condition, it relies on the break statement for
termination.

When a break or continue lies within a nested loop, it applies only to the innermost
loop. This convention generally has the desired effect. As an example, consider
how we could expand [Figure 13-5](./13_5_terminating_a_loop_break_and_continue.ipynb#fig-13-5) to perform a separate sensitivity analysis on each
avail[t]. The repeat loop would be nested in a for {t in 1..T} loop, but the
continue and break statements would apply to the inner repeat as before.

There do exist situations in which the logic of a script requires breaking out of multiple
loops. In the script of [Figure 13-5](./13_5_terminating_a_loop_break_and_continue.ipynb#fig-13-5), for instance, we can imagine that instead of stopping
when `Time[3].dual` is zero,

```ampl
if Time[3].dual = 0 then break;
```

we want to stop when Time[t].dual falls below 2700 for any t. It might seem that
one way to implement this criterion is:

```ampl
for {t in 1..T}
       if Time[t].dual < 2700 then break;
```

This statement does not have the desired effect, however, because break applies only to
the inner for loop that contains it, rather than to the outer repeat loop as we desire. In
such situations, we can give a name to a loop, and break or continue can specify by
name the loop to which it should apply. Using this feature, the outer loop in our example
could be named sens_loop:

```ampl
repeat sens_loop {
```

and the test for termination inside it could refer to its name:

```ampl
for {t in 1..T}
       if Time[t].dual < 2700 then break sens_loop;
```

The loop name appears right after repeat or for, and after break or continue.